# Cell-type FDR (reference vs query)

Compare a **query** annotation in `adata.obs` against a **reference** annotation (e.g. scanpy manual labels).

For each label predicted by the query method:

- **TP**: reference and query both equal that label
- **FP**: query equals the label but reference does not
- **FDR** = FP / (TP + FP); **precision** = 1 − FDR

**API:** `celltype_fdr(adata, ref_col, query_col)` and `plot_celltype_fdr(fdr_df, ...)`.

Demo loads [`scanpy_cluster_cellbased.h5ad`](scanpy_cluster_cellbased.h5ad) (scanpy reference: `cell_type_scanpy`; query: `cell_type_rule`).


In [ ]:
from __future__ import annotations

from typing import Literal

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


In [ ]:
def _micro_precision(fdr_df: pd.DataFrame) -> float:
    """Aggregate precision: sum(TP) / sum(n_predicted)."""
    if fdr_df.empty:
        return float("nan")
    tp = fdr_df["TP"].sum()
    n_pred = fdr_df["n_predicted"].sum()
    return float(tp / n_pred) if n_pred else float("nan")


def celltype_fdr(
    adata: ad.AnnData,
    ref_col: str,
    query_col: str,
    *,
    exclude_query_labels: set[str] | None = None,
) -> pd.DataFrame:
    """Per query-predicted label: TP/FP vs reference (FDR = FP / (TP+FP))."""
    missing = [c for c in (ref_col, query_col) if c not in adata.obs.columns]
    if missing:
        raise KeyError(f"Missing adata.obs columns: {missing}")

    exclude_query_labels = exclude_query_labels or set()
    ref = adata.obs[ref_col].astype(str)
    query = adata.obs[query_col].astype(str)

    results: list[dict] = []
    for cell_type in sorted(query.dropna().unique()):
        if cell_type in exclude_query_labels:
            continue
        truth_pos = ref == cell_type
        pred_pos = query == cell_type
        tp = int((truth_pos & pred_pos).sum())
        fp = int((~truth_pos & pred_pos).sum())
        n_predicted = tp + fp
        fdr = fp / n_predicted if n_predicted > 0 else float("nan")
        results.append(
            {
                "cell_type": cell_type,
                "TP": tp,
                "FP": fp,
                "n_predicted": n_predicted,
                "FDR": fdr,
                "precision": 1 - fdr if n_predicted > 0 else float("nan"),
            }
        )

    return (
        pd.DataFrame(results)
        .sort_values("FDR", na_position="last")
        .reset_index(drop=True)
    )


In [ ]:
def plot_celltype_fdr(
    fdr_df: pd.DataFrame,
    *,
    ref_name: str,
    query_name: str,
    metric: Literal["fdr", "precision"] = "precision",
    ax=None,
    figsize: tuple[float, float] | None = None,
    color: str = "#2a9d8f",
):
    """Horizontal bar chart of per-type FDR or precision for the query vs reference."""
    if fdr_df.empty:
        raise ValueError("fdr_df is empty; nothing to plot")

    if metric == "precision":
        plot_df = fdr_df.sort_values("precision", ascending=True, na_position="first")
        values = plot_df["precision"]
        xlabel = "Precision (1 − FDR)"
        title = f"{query_name}: precision vs {ref_name} reference"
        ref_line = 0.5
    elif metric == "fdr":
        plot_df = fdr_df.sort_values("FDR", ascending=True, na_position="first")
        values = plot_df["FDR"]
        xlabel = "FDR"
        title = f"{query_name}: FDR vs {ref_name} reference"
        ref_line = None
    else:
        raise ValueError(f"metric must be 'fdr' or 'precision', got {metric!r}")

    n_types = len(plot_df)
    if figsize is None:
        figsize = (10, max(4, 0.45 * n_types))

    if ax is None:
        _, ax = plt.subplots(figsize=figsize)

    y = np.arange(n_types)
    ax.barh(y, values, color=color, height=0.7)
    ax.set_yticks(y)
    ax.set_yticklabels(plot_df["cell_type"])
    ax.set_xlim(0, 1.05)
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    if ref_line is not None:
        ax.axvline(ref_line, color="gray", linestyle=":", linewidth=0.8)
    plt.tight_layout()
    return ax, plot_df


## Demo (Dataset 04)

Reads `adata` from `/mnt/scratch2/Maycon/Hackathon/SJ_BioHack_2026/KIDS26-Team18/data/Dataset_04/processed_data/scanpy_cluster_cellbased.h5ad`. Adjust `REF_COL` / `QUERY_COL` if your column names differ.


In [ ]:
from pathlib import Path
from IPython.display import display

H5AD_PATH = Path("/mnt/scratch2/Maycon/Hackathon/SJ_BioHack_2026/KIDS26-Team18/data/Dataset_04/processed_data/scanpy_cluster_cellbased.h5ad")

# --- configure ---
REF_COL = "cell_type_scanpy"
QUERY_COL = "cell_type_rule"
EXCLUDE_QUERY = {"Unassigned", "Ambiguous"}
REF_NAME = "scanpy"
QUERY_NAME = "rule-based"

adata = ad.read_h5ad(H5AD_PATH)
print(f"Loaded {adata.n_obs:,} cells from {H5AD_PATH.name}")

missing_cols = [c for c in (REF_COL, QUERY_COL) if c not in adata.obs.columns]
if missing_cols:
    raise KeyError(
        f"{H5AD_PATH.name} is missing adata.obs columns: {missing_cols}. "
        "Run 01_celltypeing_compare.ipynb through rule assignment and save with write_h5ad."
    )

fdr_df = celltype_fdr(
    adata,
    REF_COL,
    QUERY_COL,
    exclude_query_labels=EXCLUDE_QUERY,
)

micro = _micro_precision(fdr_df)
macro = float(fdr_df["precision"].mean()) if len(fdr_df) else float("nan")
print(f"Micro precision ({QUERY_NAME} vs {REF_NAME}): {micro:.3f}")
print(f"Macro precision (mean per predicted type):     {macro:.3f}")
print(
    f"Higher precision → {QUERY_NAME} agrees more with {REF_NAME} when it assigns a label."
)

display(
    fdr_df.rename(columns={"precision": f"precision_vs_{REF_NAME}"})
)

n_types = max(len(fdr_df), 1)
figsize = (10, max(4, 0.45 * n_types))

plot_celltype_fdr(
    fdr_df,
    ref_name=REF_NAME,
    query_name=QUERY_NAME,
    metric="precision",
    figsize=figsize,
)
plt.show()

plot_celltype_fdr(
    fdr_df,
    ref_name=REF_NAME,
    query_name=QUERY_NAME,
    metric="fdr",
    figsize=figsize,
    color="#457b9d",
)
plt.show()


## Smoke test (synthetic)

Quick check that TP/FP/FDR match hand-counted values.


In [ ]:
def _assert_fdr_row(df, cell_type, tp, fp):
    row = df.loc[df["cell_type"] == cell_type].iloc[0]
    assert row["TP"] == tp and row["FP"] == fp, (cell_type, row.to_dict())
    n = tp + fp
    assert abs(row["FDR"] - fp / n) < 1e-9
    assert abs(row["precision"] - tp / n) < 1e-9


obs = pd.DataFrame(
    {
        "ref": ["A", "A", "A", "B", "B", "C"],
        "query": ["A", "A", "B", "B", "B", "C"],
    }
)
tiny = ad.AnnData(np.zeros((6, 1)), obs=obs)

out = celltype_fdr(tiny, "ref", "query")
_assert_fdr_row(out, "A", tp=2, fp=0)
_assert_fdr_row(out, "B", tp=2, fp=1)
_assert_fdr_row(out, "C", tp=1, fp=0)
print("Synthetic celltype_fdr: OK")
print(out.to_string(index=False))
